# Simulation study of American-style Asian put options
In this notebook, a run of the simulation study is demonstrated under the rough Bergomi model using the adapted primal-dual framework with a neural network implementation. As such, this script illustrates how the simulations of the thesis are run. It is built to enable anyone to run the script for whichever configuration suits them best.

In case you are interested in the baseline configuration, you can get all respective variables by running the following command:

In [ ]:
# from thesis_code.configs.baseline_config import *

In case you are interested in a different set of parameters, feel free to choose any of your liking below:

In [ ]:
T = 1.0 # maturity
N_exercise = 12 # number of early exercise dates
r = 0.05 # risk-free interest rate

eta = 1.9 # vol-of-vol
xi = 0.09 # initial forward variance (constant curve)
rho = -0.9 # correlation between Brownian drivers of variance and level price
X0 = 1 # starting value of level price
H = 0.07 # Hurst parameter
N_grid = 48 # number of finer grid time steps

M_train_primal = 2**18 # primal train sample size
M_test_primal = 2**18 # primal pricing sample size
M_train_dual = 2**18 # dual train sample size
M_test_dual = 2**18 # dual pricing sample size
log_prices = True # whether log prices or level prices are used

# primal neural network architecture
primal_layers=3
primal_nodes=90
primal_epochs=15
primal_batch_size=2**8
primal_learning_rate=0.001
primal_activation='tanh'
primal_dropout=0
primal_polynomials=0
primal_state_spec=["average"]
primal_regularizer = "Ridge"
primal_regularizer_alpha = 0.0

# dual neural network architecture
dual_layers=6
dual_nodes=90
dual_epochs=15
dual_batch_size=2**8
dual_learning_rate=0.001
dual_activation='relu'
dual_dropout=0
dual_polynomials=0
dual_state_spec=["average"]
dual_regularizer = "Ridge"
dual_regularizer_alpha = 0.0

# signature representation
# signature_spec can be ["standard signature", "basis words signature", "log signature"]
# Note: basis words signature is the label for the Lyndon word signature components
signature_spec = "basis words signature"
# signature input process specification:
# signature_lift can be ["baseline","volatility","qv_volatility","payoff"]
signature_lift = ["qv_volatility","volatility"]
K_trunc = 3 # truncation level
# strikes = [0.7,0.8,0.9,1.0,1.1,1.2]

For this illustrative purpose, we will just run the simulation for one strike price. If you prefer several strike prices, you can adjust the script by adding a loop below. You are welcome to take a look at scripts/main_rB_colab.py

In [ ]:
strike = 1.20

In case you prefer a shorter runtime (than approximately 10 minutes) per strike, you could reduce the N_grid variable, or the samples, or the epochs, or increase the batch size.

To price an American-style Asian put option, we need to define the payoff function. Here the average is taken across the finer time grid prices.

In [ ]:
import numpy as np

def asian_put_payoff(paths):
    avg_price = np.cumsum(paths, axis=1) / np.arange(1, np.shape(paths)[1]+1)
    return np.maximum(strike - avg_price, 0.0)

Next, we need to create the sample paths:

In [ ]:
from thesis_code.deep_pricer import Pricer

pricer = Pricer(N_exercise=N_exercise, T=T, r=r, payoff=asian_put_payoff, dim_W=2)
paths = pricer.create_paths(M_train_primal, M_test_primal, M_train_dual, M_test_dual, N_grid, X0, H, xi, eta, rho)

To create the signature and the other input objects, call the create_samples function as below:

In [ ]:
signature_lift_primal = signature_lift[0]
signature_lift_dual = signature_lift[1]

pricer.create_samples(*paths,
                    signature_spec, signature_lift_primal, signature_spec, signature_lift_dual, K_trunc, strike, log_prices)


Now we are ready to train the neural networks using the train function

In [ ]:
pricer.train(primal_layers, primal_nodes, primal_epochs, primal_batch_size, primal_learning_rate, primal_activation, primal_dropout, primal_polynomials, primal_state_spec, primal_regularizer, primal_regularizer_alpha,
            dual_layers, dual_nodes, dual_epochs, dual_batch_size, dual_learning_rate, dual_activation, dual_dropout, dual_polynomials, dual_state_spec, dual_regularizer, dual_regularizer_alpha)

After training the networks, we can price new test paths to get lower and upper bound estimates as well as the duality gap

In [ ]:
lower_bound, lower_std, upper_bound, upper_std, gap = pricer.test()

The results are printed below;

In [ ]:
print(f"Results for: strike={strike}, K_trunc={K_trunc}, signature_spec={signature_spec}, signature_lift_primal={signature_lift_primal}, signature_lift_dual={signature_lift_dual}")
print(f"Lower Bound (Primal) : {lower_bound:.4f}  ±  {lower_std:.4f}")
print(f"Upper Bound (Dual)   : {upper_bound:.4f}  ±  {upper_std:.4f}")
print(f"Duality Gap          : {gap * 100:.2f}%")

You are welcome to experiment with the script and the configuration. The scripts that are repeatedly used to generate the results from the master's thesis are available in the /scripts folder.